# Machine Learning Pipeline

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pmdarima import auto_arima
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_squared_error

In [ ]:
TICKER = "BBCA.JK"
TARGET_COL = "open"
TRAIN_SIZE_RATIO = 0.9

ARIMA_START_P = 0
ARIMA_MAX_P = 3
ARIMA_START_Q = 0
ARIMA_MAX_Q = 3
ARIMA_D = 1

SARIMA_START_P = 0
SARIMA_MAX_P = 2
SARIMA_START_Q = 0
SARIMA_MAX_Q = 2
SARIMA_D = 1

SARIMA_START_SEASONAL_P = 0
SARIMA_MAX_SEASONAL_P = 2
SARIMA_START_SEASONAL_Q = 0
SARIMA_MAX_SEASONAL_Q = 2
SARIMA_SEASONAL_D = 1
SARIMA_SEASONAL_PERIOD = 5

In [ ]:
df = pd.read_csv(f"artifact/data/{TICKER}.csv", parse_dates=["date"])
df = df.sort_values("date")
df = df.reset_index(drop=True)

df.head()

## Data Understanding

### Pengecekan Informasi Data

In [ ]:
start_date = df["date"].min()
end_date = df["date"].max()
total_days = (end_date - start_date).days

print(f"Tanggal Awal Data : {start_date.strftime('%Y-%m-%d')}")
print(f"Tanggal Akhir Data : {end_date.strftime('%Y-%m-%d')}")
print(f"Total Rentang Waktu : {total_days} Hari Kalender")
print(f"Total Hari Perdagangan Aktif : {len(df)} Hari Kerja")

### Pengecekan Data Hilang

In [ ]:
print(df.isna().sum())

### Visualisasi Harga

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df["date"], df["open"], label="Open", color="blue", alpha=0.7)
plt.plot(df["date"], df["close"], label="Close", color="orange", alpha=0.7)
plt.title(f"Tren Pergerakan Harga Saham {TICKER} (Open vs Close)")
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rupiah)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## Data Preparation

### Pembagian Data Latih dan Uji

In [ ]:
train_size = int(len(df) * TRAIN_SIZE_RATIO)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print(f"Jumlah Data Train : {len(train_df)}")
print(f"Jumlah Data Test  : {len(test_df)}")

### Dekomposisi Seasonal

In [ ]:
df_decom = df.set_index("date")
decomposition = seasonal_decompose(df_decom[TARGET_COL], model="additive", period=5)

fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.suptitle(f"Dekomposisi Seasonal Harga Saham {TICKER}", y=1.02)
plt.show()

## Modelling & Evaluation

In [ ]:
train_y = train_df[TARGET_COL].values
test_y = test_df[TARGET_COL].values

### Model ARIMA

In [ ]:
print("Memulai pencarian parameter ARIMA optimal...")

arima_model = auto_arima(
    train_y,
    start_p=ARIMA_START_P,
    max_p=ARIMA_MAX_P,
    start_q=ARIMA_START_Q,
    max_q=ARIMA_MAX_Q,
    d=ARIMA_D,
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
)

arima_forecast = arima_model.predict(n_periods=len(test_y))

best_arima_order = arima_model.order
best_arima_mse = mean_squared_error(test_y, arima_forecast)
best_arima_mape = np.mean(np.abs((test_y - arima_forecast) / test_y)) * 100

print(f"Best Order: {best_arima_order}, MAPE: {best_arima_mape:.2f}%")

### Model SARIMA

In [ ]:
print("Memulai pencarian parameter SARIMA optimal...")

sarima_model = auto_arima(
    train_y,
    start_p=SARIMA_START_P,
    max_p=SARIMA_MAX_P,
    start_q=SARIMA_START_Q,
    max_q=SARIMA_MAX_Q,
    d=SARIMA_D,
    seasonal=True,
    m=SARIMA_SEASONAL_PERIOD,
    start_P=SARIMA_START_SEASONAL_P,
    max_P=SARIMA_MAX_SEASONAL_P,
    start_Q=SARIMA_START_SEASONAL_Q,
    max_Q=SARIMA_MAX_SEASONAL_Q,
    D=SARIMA_SEASONAL_D,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
)

sarima_forecast = sarima_model.predict(n_periods=len(test_y))

best_sarima_order = sarima_model.order
best_sarima_seasonal = sarima_model.seasonal_order
best_sarima_mse = mean_squared_error(test_y, sarima_forecast)
best_sarima_mape = np.mean(np.abs((test_y - sarima_forecast) / test_y)) * 100

print(f"Best Order: {best_sarima_order}x{best_sarima_seasonal}, MAPE: {best_sarima_mape:.2f}%")

### Visualisasi Hasil

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_df["date"], train_df[TARGET_COL], color="blue", label="Train Data")
plt.plot(test_df["date"], test_df[TARGET_COL], color="green", label="Test Data")
plt.plot(test_df["date"], arima_forecast, color="red", label="ARIMA Forecast")

plt.title(f"Forecast ARIMA {best_arima_order} - {TICKER}")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_df["date"], train_df[TARGET_COL], color="blue", label="Train Data")
plt.plot(test_df["date"], test_df[TARGET_COL], color="green", label="Test Data")
plt.plot(test_df["date"], sarima_forecast, color="red", label="SARIMA Forecast")

plt.title(f"Forecast SARIMA {best_sarima_order}x{best_sarima_seasonal} - {TICKER}")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

### Menyimpan Model

In [ ]:
full_data = df[TARGET_COL].values

print(f"Melatih ulang ARIMA {best_arima_order} pada seluruh data...")
final_arima = auto_arima(
    full_data,
    start_p=best_arima_order[0],
    max_p=best_arima_order[0],
    d=best_arima_order[1],
    start_q=best_arima_order[2],
    max_q=best_arima_order[2],
    seasonal=False,
    suppress_warnings=True,
    error_action="ignore",
)

print(f"Melatih ulang SARIMA {best_sarima_order}x{best_sarima_seasonal} pada seluruh data...")
final_sarima = auto_arima(
    full_data,
    start_p=best_sarima_order[0],
    max_p=best_sarima_order[0],
    d=best_sarima_order[1],
    start_q=best_sarima_order[2],
    max_q=best_sarima_order[2],
    start_P=best_sarima_seasonal[0],
    max_P=best_sarima_seasonal[0],
    D=best_sarima_seasonal[1],
    start_Q=best_sarima_seasonal[2],
    max_Q=best_sarima_seasonal[2],
    m=SARIMA_SEASONAL_PERIOD,
    seasonal=True,
    suppress_warnings=True,
    error_action="ignore",
)

if best_sarima_mape < best_arima_mape:
    print(f"Model Performa Terbaik: SARIMA (MAPE: {best_sarima_mape:.2f}%)")
else:
    print(f"Model Performa Terbaik: ARIMA (MAPE: {best_arima_mape:.2f}%)")

In [ ]:
output_dir = Path("artifact/model")
output_dir.mkdir(parents=True, exist_ok=True)

path_arima = output_dir / f"{TICKER}_ARIMA.pkl"
path_sarima = output_dir / f"{TICKER}_SARIMA.pkl"

with open(path_arima, "wb") as f:
    pickle.dump(final_arima, f)

with open(path_sarima, "wb") as f:
    pickle.dump(final_sarima, f)

print(f"Model ARIMA berhasil disimpan ke : {path_arima}")
print(f"Model SARIMA berhasil disimpan ke: {path_sarima}")